# 🚌 CTA Ingestion & Real-Time Live Streaming
This notebook demonstrates:
1. Downloading & extracting CTA GTFS static timetables (`routes.txt`, `stops.txt`, `stop_times.txt`)
2. Ingesting Chicago Traffic Tracker arterial speed estimates
3. Streaming & parsing live vehicle locations, speeds, and delays via CTA GTFS-RT / Bus Tracker API
4. Persisting live snapshots into `data/raw/live_snapshots/` for time-series logging

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

from src.data_loader import load_cta_gtfs, load_traffic_congestion, load_bus_ridership
from src.live_feed import CTALiveTracker

## 1. Load GTFS Static Network Feeds

In [ ]:
gtfs_tables = load_cta_gtfs()
print(f"Loaded GTFS tables: {list(gtfs_tables.keys())}")
if 'routes' in gtfs_tables:
    display(gtfs_tables['routes'].head())
if 'stops' in gtfs_tables:
    display(gtfs_tables['stops'].head())

## 2. Ingest Real-Time Vehicle Tracking (CTA Bus Tracker / GTFS-RT)

In [ ]:
tracker = CTALiveTracker()
# Poll live active vehicles for key Chicago corridors
df_live = tracker.get_live_vehicles(routes=['22', '9', '66', '49', '79'])
print(f"Active vehicles fetched: {len(df_live)}")
display(df_live[['vid', 'rt', 'spd', 'delay_min', 'delay_status', 'lat', 'lon']].head(10))

## 3. Save Real-Time Snapshot

In [ ]:
snapshot_file = tracker.save_snapshot(df_live)
print(f"Snapshot logged successfully to {snapshot_file}")